# 13 - Gold 2: fact_network_activity
**Grain:** one row per `measurement_id` (its current state). The business key `measurement_id` is unique.

**Incremental load:** a MERGE on `measurement_id`. New readings are inserted, readings whose status, outage minutes or
secondary-event summary changed are updated, and everything else is left alone. Running the same input again changes **no rows** (TC03).

**Outage minutes per reading:** for a reading whose current status is FAILED, minutes = time until the same tower's next reading.
Summed over a run of FAILED readings this equals the outage interval length, so facts, marts and Silver intervals reconcile.
A FAILED reading with no later reading is an open outage: 0 minutes and `is_open_outage = true`.

In [0]:
BRONZE = "capstone6_telecom.bronze"
SILVER = "capstone6_telecom.silver"
GOLD   = "capstone6_telecom.gold"
cat    = GOLD   # this notebook writes to the gold schema

## 1. Source view (Silver enriched + tower-next-reading + dimension keys)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW _fact_src AS
SELECT e.measurement_id, e.event_ts, CAST(e.event_ts AS DATE) AS event_date,
       CAST(date_format(e.event_ts, 'yyyyMMdd') AS INT) AS date_key,
       r.region_key, t.tower_key, e.region_id, e.tower_id, e.operator_id, e.network_type,
       e.network_status AS initial_status, e.current_status, e.status_sequence,
       e.latency_ms, e.packet_loss_pct, e.latency_outlier_flag,
       (e.current_status = 'FAILED') AS is_outage_reading,
       CASE WHEN e.current_status = 'FAILED' AND e.next_ts IS NOT NULL
            THEN (unix_timestamp(e.next_ts) - unix_timestamp(e.event_ts)) / 60.0
            ELSE 0.0 END AS outage_minutes,
       (e.current_status = 'FAILED' AND e.next_ts IS NULL) AS is_open_outage,
       e.has_alert, e.secondary_event_count,
       current_timestamp() AS updated_at
FROM (SELECT *, LEAD(event_ts) OVER (PARTITION BY tower_id ORDER BY event_ts, measurement_id) AS next_ts
      FROM {SILVER}.silver_network_enriched) e
LEFT JOIN {cat}.dim_region r ON e.region_id = r.region_id
LEFT JOIN {cat}.dim_tower  t ON e.tower_id  = t.tower_id
""")

## 2. Create the table once, then MERGE

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {cat}.fact_network_activity AS SELECT * FROM _fact_src WHERE 1 = 0")

spark.sql(f"""
MERGE INTO {cat}.fact_network_activity t
USING _fact_src s
ON t.measurement_id = s.measurement_id
WHEN MATCHED AND NOT (t.current_status <=> s.current_status
                  AND t.status_sequence <=> s.status_sequence
                  AND t.outage_minutes <=> s.outage_minutes
                  AND t.is_open_outage <=> s.is_open_outage
                  AND t.has_alert <=> s.has_alert
                  AND t.secondary_event_count <=> s.secondary_event_count)
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

## 3. Checks

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS fact_rows,                                          -- expect 11,500 (= silver events)
       COUNT(DISTINCT measurement_id) AS unique_keys,                  -- must equal fact_rows
       SUM(CASE WHEN region_key IS NULL OR tower_key IS NULL OR date_key IS NULL THEN 1 ELSE 0 END) AS missing_keys,  -- 0
       SUM(CASE WHEN is_outage_reading THEN 1 ELSE 0 END) AS outage_readings,   -- 2,755
       SUM(CASE WHEN is_open_outage THEN 1 ELSE 0 END) AS open_outage_readings
FROM {cat}.fact_network_activity"""))

## 4. What did the last MERGE do? (run this notebook twice: the second run must insert 0 and update 0)

In [0]:
from pyspark.sql import functions as F

h = spark.sql(f"DESCRIBE HISTORY {cat}.fact_network_activity")
display(h.select("version", "timestamp", "operation",
                 F.col("operationMetrics")["numTargetRowsInserted"].alias("inserted"),
                 F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated"))
         .orderBy(F.col("version").desc()).limit(5))